In [4]:

import pandas as pd
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm


sabdab_summary_file = "~/Downloads/20250723_0200598_summary.tsv"
MAX_WORKERS = 20

def fetch_chothia_cdr(pdb_id, chain_id, loop):
    base_url = "https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabdab/cdrviewer/"
    for cid in [chain_id, chain_id.lower()]:
        url = f"{base_url}?pdb={pdb_id}&loop={loop}&chain={cid}&CDRdef=chothia"
        try:
            r = requests.get(url, timeout=10)
            soup = BeautifulSoup(r.text, "html.parser")
            table = soup.find("table")
            if not table:
                continue
            for row in table.find_all("tr"):
                if "Sequence" in row.text:
                    cells = row.find_all("td")
                    if len(cells) >= 2:
                        return cells[-1].text.strip()
        except:
            continue
    return None

# Read the .tsv file while handling bad lines, extra whitespace, and inconsistent spacing
df = pd.read_csv(sabdab_summary_file, 
                 sep='\t', 
                 engine='python', 
                 skip_blank_lines=True, 
                 on_bad_lines='skip')


df['Hchain'] = df['Hchain'].fillna('').astype(str)
df['Lchain'] = df['Lchain'].fillna('').astype(str)

loops = ["H1","H2","H3","L1","L2","L3"]

def _task(args):
    idx, row, loop = args

    chain = row['Hchain'] if loop.startswith('H') else row['Lchain']
    if not chain:
        return idx, loop, None

    query_loop = f"CDR{loop}"
    seq = fetch_chothia_cdr(row['pdb'], chain, query_loop)
    return idx, loop, seq

tasks = [
    (idx, row, loop)
    for idx, row in df.iterrows()
    for loop in loops
]


with ThreadPoolExecutor(max_workers=MAX_WORKERS) as exe:
    for idx, loop, seq in tqdm(exe.map(_task, tasks), total=len(tasks)):
        df.at[idx, loop] = seq
df

  0%|          | 0/8628 [00:00<?, ?it/s]

,pdb,Hchain,Lchain,model,antigen_chain,antigen_type,antigen_het_name,antigen_name,short_header,date,...,delta_g,affinity_method,temperature,pmid,H1,H2,H3,L1,L2,L3
0,9cr9,A,B,0,C,peptide,NaN,circumsporozoite protein (npna)3 repeat region...,IMMUNE SYSTEM,07/09/25,...,NaN,NaN,NaN,NaN,GFTFSSY,WYDGSN,DGGHSTSWSNAFDI,RASQSISSWLA,KASSLES,QQYNSYWT
1,9cr9,H,L,0,P,peptide,NaN,circumsporozoite protein (npna)3 repeat region...,IMMUNE SYSTEM,07/09/25,...,NaN,NaN,NaN,NaN,GFTFSSY,WYDGSN,DGGHSTSWSNAFDI,RASQSISSWLA,KASSLES,QQYNSYWT
2,8vwe,H,L,0,C,peptide,NaN,transmembrane protein gp41 iqn17 peptide,VIRAL PROTEIN/IMMUNE SYSTEM,07/02/25,...,NaN,NaN,NaN,NaN,GDTFSSY,IPLFGT,DNPTFGAADS,RASEGIYHWLA,KASSLAS,QQYSNYPLT
3,9j8a,H,L,0,P,peptide,NaN,sulfated peptide from ccr5,IMMUNE SYSTEM,07/02/25,...,NaN,NaN,NaN,NaN,GFSLSSY,GSGGN,GGINFGI,QSSQSVYSKNWLS,SASTLDS,QGTYGDASA
4,8s6k,A,A,0,M,peptide,NaN,mucin-1 subunit alpha,IMMUNE SYSTEM,06/25/25,...,NaN,NaN,NaN,NaN,GYIFADH,SPGNDD,SLPGTFDY,KASQDVGTAVA,WASTRHT,QQYSSYPLT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1433,6bzy,H,L,0,B,peptide,NaN,e2 as412 peptide,IMMUNE SYSTEM,12/26/17,...,NaN,NaN,NaN,NaN,GFSLTSY,WSAGN,APIYYDYTWFAY,RASESVDNYGISFMN,AASNQGS,QQSKEVPYT
1434,6o2c,A,B,0,C,peptide,NaN,circumsporozoite protein,IMMUNE SYSTEM,02/22/19,...,NaN,NaN,NaN,NaN,GFTFGDY,RSKANGGR,VELGSSWSLGY,RASQSVSSTYLA,GASSRAT,QQYGSSPWT
1435,4hpo,H,L,0,P,peptide,NaN,envelope glycoprotein gp160,Immune System/Viral Protein,10/24/12,...,NaN,NaN,NaN,NaN,GYRFTSY,YPGDFD,LGGRYYHDSSGYYYLDY,TRSSGSVASDYVQ,EDNQRPS,QSYDNSSWV
1436,7a48,A,,0,B,peptide,NaN,aph colied-coil,DE NOVO PROTEIN,08/19/20,...,NaN,NaN,NaN,NaN,GRTFNPY,TWIGGT,DKDNTGYNY,None,None,None


In [6]:
df["delta_g"].sum

<bound method Series.sum of 0      NaN
1      NaN
2      NaN
3      NaN
4      NaN
        ..
1433   NaN
1434   NaN
1435   NaN
1436   NaN
1437   NaN
Name: delta_g, Length: 1438, dtype: float64>